# Telecom Network Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Telecom industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Telecom Network Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about telecom network operations, trouble tickets,
network performance, outage events, SLA compliance, field dispatches,
technician wellness, and customer satisfaction using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables:
   - dim_engineers: engineer_id, name, role, department, hire_date, certifications, specialty, region
   - dim_enterprise_customers: customer_id, name, industry, contract_type, sla_tier, account_manager, region
   - dim_equipment: equipment_id, name, type, manufacturer, model, install_date, status, network_segment_id
   - dim_network_segments: segment_id, name, type, region, capacity, technology, status
   - dim_service_areas: area_id, name, region, state, population, coverage_type, towers
   - dim_ticket_types: ticket_type_id, name, category, priority_default, sla_hours, escalation_path

   Batch fact tables:
   - fact_customer_satisfaction: survey_id, customer_id, engineer_id, date, overall_score, response_score, resolution_score, nps
   - fact_field_dispatches: dispatch_id, engineer_id, ticket_id, timestamp, travel_time_min, onsite_time_min, resolution_flag, equipment_id
   - fact_field_location: location_id, engineer_id, timestamp, lat, lon, service_area_id, activity_type
   - fact_network_performance: perf_id, segment_id, date, latency_ms, throughput_mbps, packet_loss_pct, uptime_pct, jitter_ms
   - fact_nms_interactions: interaction_id, engineer_id, timestamp, equipment_id, action, duration_seconds, alarm_acknowledged, screen
   - fact_outage_events: outage_id, segment_id, service_area_id, timestamp, duration_min, affected_customers, root_cause, severity

   Event fact tables:
   - fact_rca_documents: rca_id, outage_id, engineer_id, date, quality_score, completeness_pct, root_cause_verified, revision_count
   - fact_sla_alerts: alert_id, customer_id, ticket_id, timestamp, sla_metric, threshold, actual_value, breach_flag
   - fact_technician_wellness: survey_id, engineer_id, date, workload_score, overtime_hours, burnout_risk, fatigue_level, call_volume
   - fact_ticket_escalations: escalation_id, ticket_id, from_engineer_id, to_engineer_id, timestamp, reason, doc_completeness_pct, pending_items
   - fact_ticket_quality: quality_id, engineer_id, ticket_type_id, date, quality_score, completeness_pct, errors_found, resolution_time_min
   - fact_trouble_tickets: ticket_id, customer_id, engineer_id, ticket_type_id, open_date, close_date, status, priority, segment_id

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse.
3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.
4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (KQL).
   Event tables: rca_documents, sla_alerts, technician_wellness, ticket_escalations, ticket_quality, trouble_tickets
   Streaming tables: customer_impact, field_dispatch, network_alarms, performance_metrics, ticket_activity

== KEY RELATIONSHIPS ==
- dim_engineers.engineer_id → fact tables on engineer_id (also from_engineer_id, to_engineer_id)
- dim_enterprise_customers.customer_id → fact_customer_satisfaction, fact_sla_alerts, fact_trouble_tickets
- dim_network_segments.segment_id → fact_network_performance, fact_outage_events, fact_trouble_tickets
- dim_equipment.equipment_id → fact_nms_interactions, fact_field_dispatches
- dim_service_areas.area_id → fact_outage_events, fact_field_location
- dim_ticket_types.ticket_type_id → fact_trouble_tickets, fact_ticket_quality

== EXAMPLE QUERIES ==

Q: What is the network performance by segment?
→ Query fact_network_performance, GROUP BY segment_id, AVG(latency_ms), AVG(uptime_pct), join dim_network_segments.

Q: Show trouble ticket summary.
→ Query fact_trouble_tickets, GROUP BY status, priority, COUNT(*), join dim_ticket_types.

Q: Which segments have outage events?
→ Query fact_outage_events, GROUP BY segment_id, COUNT(*), SUM(affected_customers), join dim_network_segments.

Q: Show SLA compliance.
→ Query fact_sla_alerts, GROUP BY sla_metric, SUM(breach_flag), COUNT(*).

Q: Show field dispatch metrics.
→ Query fact_field_dispatches, GROUP BY engineer_id, AVG(travel_time_min), AVG(onsite_time_min), join dim_engineers.

Q: Show ticket quality by engineer.
→ Query fact_ticket_quality, GROUP BY engineer_id, AVG(quality_score), join dim_engineers.

Q: Show customer satisfaction by account.
→ Query fact_customer_satisfaction, GROUP BY customer_id, AVG(overall_score), join dim_enterprise_customers.

Q: Show ticket escalation patterns.
→ Query fact_ticket_escalations, GROUP BY from_engineer_id, AVG(doc_completeness_pct), join dim_engineers.

Q: Show technician wellness.
→ Query fact_technician_wellness, GROUP BY engineer_id, AVG(workload_score), join dim_engineers.

Q: Show NMS interactions by action.
→ Query fact_nms_interactions, GROUP BY action, COUNT(*), AVG(duration_seconds).
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Telecom Network Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on network outages, SLA breaches,
ticket escalations, equipment alarms, field dispatch delays, and technician wellness.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis.
   Key operational tables:
   - fact_outage_events: CRITICAL if severity = 'Critical', affected_customers > 1000
   - fact_sla_alerts: CRITICAL if breach_flag = true
   - fact_technician_wellness: CRITICAL if burnout_risk = true, fatigue_level > 8
   - fact_network_performance: latency_ms > 100, uptime_pct < 99, packet_loss_pct > 2
   - fact_trouble_tickets: status = 'Open', priority = 'Critical'
   - fact_field_dispatches: resolution_flag = false, travel_time_min > 120
   - fact_ticket_escalations: doc_completeness_pct < 80
   - fact_ticket_quality: quality_score < 70, errors_found > 5
   - fact_nms_interactions: alarm_acknowledged = false
   - fact_customer_satisfaction: overall_score < 5, nps < 0
   - fact_rca_documents: quality_score < 70, root_cause_verified = false

   Dimension tables: dim_engineers, dim_enterprise_customers, dim_network_segments, dim_equipment, dim_service_areas, dim_ticket_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Streaming tables:
   - network_alarms: alarm_id, equipment_id, timestamp, alarm_type, severity, acknowledged, segment_id
     → CRITICAL: severity = 'Critical', acknowledged = false
   - performance_metrics: reading_id, segment_id, timestamp, latency_ms, throughput_mbps, packet_loss_pct
     → CRITICAL: latency_ms > 100, packet_loss_pct > 2
   - customer_impact: impact_id, customer_id, timestamp, service_type, impact_level, affected_services
     → CRITICAL: impact_level = 'Critical'
   - ticket_activity: activity_id, ticket_id, timestamp, action, engineer_id, status
     → Monitor for stale tickets, slow resolution
   - field_dispatch: dispatch_id, engineer_id, timestamp, ticket_id, status, location
     → Monitor for dispatch delays

== ALERTING THRESHOLDS ==
- Network: latency_ms > 100, uptime_pct < 99, packet_loss_pct > 2
- Outages: severity = 'Critical', affected_customers > 1000
- SLA: breach_flag = true
- Equipment: unacknowledged critical alarms
- Tickets: priority = 'Critical', status = 'Open'
- Personnel: burnout_risk = true, fatigue_level > 8
- Customer: impact_level = 'Critical'
- Dispatch: travel_time_min > 120, resolution_flag = false

== EXAMPLE QUERIES ==

Q: Are there critical network outages?
→ Query fact_outage_events WHERE severity = 'Critical', join dim_network_segments.

Q: Which segments have poor performance?
→ Query fact_network_performance WHERE uptime_pct < 99, join dim_network_segments.

Q: Show SLA breaches.
→ Query fact_sla_alerts WHERE breach_flag = true, join dim_enterprise_customers.

Q: Show real-time network alarms.
→ KQL: network_alarms | where severity == 'Critical' and acknowledged == false | project equipment_id, alarm_type, segment_id

Q: Show customer impact.
→ KQL: customer_impact | where impact_level == 'Critical' | project customer_id, service_type, affected_services

Q: Show network performance metrics.
→ KQL: performance_metrics | where timestamp > ago(1h) | summarize avg(latency_ms) by segment_id

Q: Which trouble tickets are critical and open?
→ Query fact_trouble_tickets WHERE status = 'Open' AND priority = 'Critical', join dim_enterprise_customers.

Q: Which technicians are fatigued?
→ Query fact_technician_wellness WHERE burnout_risk = true, join dim_engineers.

Q: Show field dispatch delays.
→ Query fact_field_dispatches WHERE travel_time_min > 120, join dim_engineers.

Q: Show unacknowledged NMS alarms.
→ KQL: network_alarms | where acknowledged == false | summarize count() by alarm_type, severity
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all telecom network operations data as SQL tables.

DIMENSION TABLES:
- dim_engineers, dim_enterprise_customers, dim_equipment, dim_network_segments, dim_service_areas, dim_ticket_types

FACT TABLES:
- fact_customer_satisfaction, fact_field_dispatches, fact_field_location, fact_network_performance,
  fact_nms_interactions, fact_outage_events

EVENT FACT TABLES:
- fact_rca_documents, fact_sla_alerts, fact_technician_wellness, fact_ticket_escalations,
  fact_ticket_quality, fact_trouble_tickets

KEY JOINS:
- dim_engineers.engineer_id → fact tables on engineer_id
- dim_enterprise_customers.customer_id → fact_customer_satisfaction, fact_sla_alerts, fact_trouble_tickets
- dim_network_segments.segment_id → fact_network_performance, fact_outage_events, fact_trouble_tickets
- dim_equipment.equipment_id → fact_nms_interactions, fact_field_dispatches""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- rca_documents, sla_alerts, technician_wellness, ticket_escalations, ticket_quality, trouble_tickets

STREAMING TABLES:
- customer_impact, field_dispatch, network_alarms, performance_metrics, ticket_activity

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_outage_events: CRITICAL on severity = 'Critical', affected_customers > 1000
- fact_sla_alerts: breach_flag = true
- fact_network_performance: latency_ms > 100, uptime_pct < 99, packet_loss_pct > 2
- fact_technician_wellness: burnout_risk = true, fatigue_level > 8
- fact_trouble_tickets: priority = 'Critical', status = 'Open'
- fact_field_dispatches: travel_time_min > 120, resolution_flag = false

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time network telemetry.

STREAMING ALERTS:
- network_alarms: severity = 'Critical', acknowledged = false
- performance_metrics: latency_ms > 100, packet_loss_pct > 2
- customer_impact: impact_level = 'Critical'
- ticket_activity: stale/slow tickets

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "What is the network performance by segment?",
            "query": """SELECT ns.name AS segment, ns.type, ns.region,\n       AVG(np.latency_ms) AS avg_latency,\n       AVG(np.throughput_mbps) AS avg_throughput,\n       AVG(np.uptime_pct) AS avg_uptime,\n       AVG(np.packet_loss_pct) AS avg_packet_loss\nFROM fact_network_performance np\nJOIN dim_network_segments ns ON np.segment_id = ns.segment_id\nGROUP BY ns.name, ns.type, ns.region\nORDER BY avg_uptime ASC"""
        },
        {
            "question": "Show trouble ticket summary.",
            "query": """SELECT tt.status, tt.priority, tty.name AS ticket_type,\n       COUNT(*) AS tickets\nFROM fact_trouble_tickets tt\nJOIN dim_ticket_types tty ON tt.ticket_type_id = tty.ticket_type_id\nGROUP BY tt.status, tt.priority, tty.name\nORDER BY tickets DESC"""
        },
        {
            "question": "Which segments have outage events?",
            "query": """SELECT ns.name AS segment, oe.root_cause, oe.severity,\n       COUNT(*) AS outages,\n       SUM(oe.affected_customers) AS total_affected,\n       AVG(oe.duration_min) AS avg_duration\nFROM fact_outage_events oe\nJOIN dim_network_segments ns ON oe.segment_id = ns.segment_id\nGROUP BY ns.name, oe.root_cause, oe.severity\nORDER BY outages DESC"""
        },
        {
            "question": "Show SLA compliance.",
            "query": """SELECT sa.sla_metric,\n       COUNT(*) AS total_alerts,\n       SUM(CASE WHEN sa.breach_flag = true THEN 1 ELSE 0 END) AS breaches\nFROM fact_sla_alerts sa\nGROUP BY sa.sla_metric\nORDER BY breaches DESC"""
        },
        {
            "question": "Show field dispatch metrics.",
            "query": """SELECT e.name, e.role,\n       COUNT(*) AS dispatches,\n       AVG(fd.travel_time_min) AS avg_travel,\n       AVG(fd.onsite_time_min) AS avg_onsite,\n       SUM(CASE WHEN fd.resolution_flag = true THEN 1 ELSE 0 END) AS resolved\nFROM fact_field_dispatches fd\nJOIN dim_engineers e ON fd.engineer_id = e.engineer_id\nGROUP BY e.name, e.role\nORDER BY dispatches DESC"""
        },
        {
            "question": "Show ticket quality by engineer.",
            "query": """SELECT e.name, e.role,\n       AVG(tq.quality_score) AS avg_quality,\n       AVG(tq.completeness_pct) AS avg_completeness,\n       SUM(tq.errors_found) AS total_errors,\n       AVG(tq.resolution_time_min) AS avg_resolution\nFROM fact_ticket_quality tq\nJOIN dim_engineers e ON tq.engineer_id = e.engineer_id\nGROUP BY e.name, e.role\nORDER BY avg_quality DESC"""
        },
        {
            "question": "Show customer satisfaction by account.",
            "query": """SELECT ec.name AS customer, ec.industry, ec.sla_tier,\n       AVG(cs.overall_score) AS avg_overall,\n       AVG(cs.resolution_score) AS avg_resolution,\n       AVG(cs.nps) AS avg_nps\nFROM fact_customer_satisfaction cs\nJOIN dim_enterprise_customers ec ON cs.customer_id = ec.customer_id\nGROUP BY ec.name, ec.industry, ec.sla_tier\nORDER BY avg_overall DESC"""
        },
        {
            "question": "Show ticket escalation patterns.",
            "query": """SELECT e.name AS from_engineer, te.reason,\n       COUNT(*) AS escalations,\n       AVG(te.doc_completeness_pct) AS avg_completeness\nFROM fact_ticket_escalations te\nJOIN dim_engineers e ON te.from_engineer_id = e.engineer_id\nGROUP BY e.name, te.reason\nORDER BY escalations DESC"""
        },
        {
            "question": "Show technician wellness.",
            "query": """SELECT e.name, e.role,\n       tw.workload_score, tw.overtime_hours,\n       tw.fatigue_level, tw.call_volume, tw.burnout_risk\nFROM fact_technician_wellness tw\nJOIN dim_engineers e ON tw.engineer_id = e.engineer_id\nORDER BY tw.workload_score DESC"""
        },
        {
            "question": "Show NMS interactions by action.",
            "query": """SELECT ni.action, ni.screen,\n       COUNT(*) AS interactions,\n       AVG(ni.duration_seconds) AS avg_duration,\n       SUM(CASE WHEN ni.alarm_acknowledged = false THEN 1 ELSE 0 END) AS unacknowledged\nFROM fact_nms_interactions ni\nGROUP BY ni.action, ni.screen\nORDER BY interactions DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which segments have the most tickets?",
            "query": """SELECT ns.name, ns.type,\n       COUNT(*) AS tickets\nFROM fact_trouble_tickets tt\nJOIN dim_network_segments ns ON tt.segment_id = ns.segment_id\nGROUP BY ns.name, ns.type\nORDER BY tickets DESC\nLIMIT 20"""
        },
        {
            "question": "Show network segments.",
            "query": """SELECT type, region, technology, status,\n       COUNT(*) AS segments\nFROM dim_network_segments\nGROUP BY type, region, technology, status\nORDER BY segments DESC"""
        },
        {
            "question": "Show service areas.",
            "query": """SELECT region, coverage_type,\n       COUNT(*) AS areas,\n       SUM(towers) AS total_towers,\n       SUM(population) AS total_population\nFROM dim_service_areas\nGROUP BY region, coverage_type\nORDER BY total_population DESC"""
        },
        {
            "question": "Show outage duration distribution.",
            "query": """SELECT severity, root_cause,\n       COUNT(*) AS outages,\n       AVG(duration_min) AS avg_duration\nFROM fact_outage_events\nGROUP BY severity, root_cause\nORDER BY avg_duration DESC"""
        },
        {
            "question": "Show enterprise customer overview.",
            "query": """SELECT industry, sla_tier, contract_type,\n       COUNT(*) AS customers\nFROM dim_enterprise_customers\nGROUP BY industry, sla_tier, contract_type\nORDER BY customers DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show network alarms.",
            "query": """network_alarms\n| where timestamp > ago(24h)\n| summarize count() by alarm_type, severity, acknowledged\n| order by count_ desc"""
        },
        {
            "question": "Show performance metrics.",
            "query": """performance_metrics\n| where timestamp > ago(1h)\n| summarize avg_latency = avg(latency_ms), avg_throughput = avg(throughput_mbps), avg_loss = avg(packet_loss_pct) by segment_id\n| order by avg_latency desc"""
        },
        {
            "question": "Show customer impact.",
            "query": """customer_impact\n| where timestamp > ago(24h)\n| summarize count() by impact_level, service_type\n| order by count_ desc"""
        },
        {
            "question": "Show ticket activity.",
            "query": """ticket_activity\n| where timestamp > ago(24h)\n| summarize count() by action, status\n| order by count_ desc"""
        },
        {
            "question": "Show field dispatches.",
            "query": """field_dispatch\n| where timestamp > ago(24h)\n| summarize count() by status, engineer_id\n| order by count_ desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Are there critical network outages?",
            "query": """SELECT ns.name AS segment, oe.root_cause,\n       oe.severity, oe.duration_min, oe.affected_customers,\n       sa.name AS service_area\nFROM fact_outage_events oe\nJOIN dim_network_segments ns ON oe.segment_id = ns.segment_id\nJOIN dim_service_areas sa ON oe.service_area_id = sa.area_id\nWHERE oe.severity = 'Critical'\nORDER BY oe.timestamp DESC"""
        },
        {
            "question": "Which segments have poor performance?",
            "query": """SELECT ns.name AS segment, ns.type,\n       np.latency_ms, np.uptime_pct, np.packet_loss_pct,\n       CASE WHEN np.uptime_pct < 95 THEN 'CRITICAL'\n            WHEN np.uptime_pct < 99 THEN 'WARNING'\n            ELSE 'OK' END AS severity\nFROM fact_network_performance np\nJOIN dim_network_segments ns ON np.segment_id = ns.segment_id\nWHERE np.uptime_pct < 99\nORDER BY np.uptime_pct ASC"""
        },
        {
            "question": "Show SLA breaches.",
            "query": """SELECT ec.name AS customer, ec.sla_tier,\n       sa.sla_metric, sa.threshold, sa.actual_value\nFROM fact_sla_alerts sa\nJOIN dim_enterprise_customers ec ON sa.customer_id = ec.customer_id\nWHERE sa.breach_flag = true\nORDER BY sa.timestamp DESC"""
        },
        {
            "question": "Which trouble tickets are critical and open?",
            "query": """SELECT ec.name AS customer, tty.name AS ticket_type,\n       tt.priority, tt.status, tt.open_date,\n       e.name AS engineer\nFROM fact_trouble_tickets tt\nJOIN dim_enterprise_customers ec ON tt.customer_id = ec.customer_id\nJOIN dim_ticket_types tty ON tt.ticket_type_id = tty.ticket_type_id\nJOIN dim_engineers e ON tt.engineer_id = e.engineer_id\nWHERE tt.status = 'Open' AND tt.priority = 'Critical'\nORDER BY tt.open_date ASC"""
        },
        {
            "question": "Which technicians are fatigued?",
            "query": """SELECT e.name, e.role, e.region,\n       tw.workload_score, tw.fatigue_level,\n       tw.overtime_hours, tw.call_volume, tw.burnout_risk\nFROM fact_technician_wellness tw\nJOIN dim_engineers e ON tw.engineer_id = e.engineer_id\nWHERE tw.burnout_risk = true\nORDER BY tw.fatigue_level DESC"""
        },
        {
            "question": "Show field dispatch delays.",
            "query": """SELECT e.name AS engineer, fd.travel_time_min,\n       fd.onsite_time_min, fd.resolution_flag\nFROM fact_field_dispatches fd\nJOIN dim_engineers e ON fd.engineer_id = e.engineer_id\nWHERE fd.travel_time_min > 120\nORDER BY fd.travel_time_min DESC"""
        },
        {
            "question": "Show ticket escalation issues.",
            "query": """SELECT e.name AS from_engineer,\n       te.reason, te.doc_completeness_pct, te.pending_items\nFROM fact_ticket_escalations te\nJOIN dim_engineers e ON te.from_engineer_id = e.engineer_id\nWHERE te.doc_completeness_pct < 80\nORDER BY te.doc_completeness_pct ASC"""
        },
        {
            "question": "Show RCA quality issues.",
            "query": """SELECT e.name AS engineer,\n       rca.quality_score, rca.completeness_pct,\n       rca.root_cause_verified, rca.revision_count\nFROM fact_rca_documents rca\nJOIN dim_engineers e ON rca.engineer_id = e.engineer_id\nWHERE rca.quality_score < 70\nORDER BY rca.quality_score ASC"""
        },
        {
            "question": "Show customer satisfaction drops.",
            "query": """SELECT ec.name AS customer, ec.sla_tier,\n       cs.overall_score, cs.resolution_score, cs.nps\nFROM fact_customer_satisfaction cs\nJOIN dim_enterprise_customers ec ON cs.customer_id = ec.customer_id\nWHERE cs.overall_score < 5\nORDER BY cs.overall_score ASC"""
        },
        {
            "question": "Show unacknowledged NMS alarms.",
            "query": """SELECT eq.name AS equipment, ni.action,\n       ni.duration_seconds, e.name AS engineer\nFROM fact_nms_interactions ni\nJOIN dim_equipment eq ON ni.equipment_id = eq.equipment_id\nJOIN dim_engineers e ON ni.engineer_id = e.engineer_id\nWHERE ni.alarm_acknowledged = false\nORDER BY ni.timestamp DESC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show critical network alarms.",
            "query": """network_alarms\n| where severity == 'Critical' and acknowledged == false\n| project equipment_id, alarm_type, segment_id\n| order by timestamp desc"""
        },
        {
            "question": "Show performance degradation.",
            "query": """performance_metrics\n| where timestamp > ago(1h)\n| where latency_ms > 100 or packet_loss_pct > 2\n| project segment_id, latency_ms, throughput_mbps, packet_loss_pct\n| order by latency_ms desc"""
        },
        {
            "question": "Show critical customer impact.",
            "query": """customer_impact\n| where impact_level == 'Critical'\n| project customer_id, service_type, affected_services\n| order by timestamp desc"""
        },
        {
            "question": "Show ticket activity.",
            "query": """ticket_activity\n| where timestamp > ago(24h)\n| summarize count() by action, status\n| order by count_ desc"""
        },
        {
            "question": "Show field dispatch status.",
            "query": """field_dispatch\n| where timestamp > ago(24h)\n| summarize count() by status, engineer_id\n| order by count_ desc"""
        },
        {
            "question": "Show network alarm trend.",
            "query": """network_alarms\n| where timestamp > ago(7d)\n| summarize count() by bin(timestamp, 1d), severity\n| order by timestamp asc"""
        },
        {
            "question": "Show latency trend.",
            "query": """performance_metrics\n| where timestamp > ago(24h)\n| summarize avg_latency = avg(latency_ms) by bin(timestamp, 1h), segment_id\n| order by timestamp asc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Telecom Data Agents

### QA Agent — `Telecom_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Network | What is the network performance by segment? |
| 2 | Tickets | Show trouble ticket summary. |
| 3 | Outages | Which segments have outage events? |
| 4 | SLA | Show SLA compliance. |
| 5 | Dispatch | Show field dispatch metrics. |
| 6 | Quality | Show ticket quality by engineer. |
| 7 | Customer | Show customer satisfaction by account. |
| 8 | Escalations | Show ticket escalation patterns. |
| 9 | Wellness | Show technician wellness. |
| 10 | NMS | Show NMS interactions by action. |
| 11 | Segments | Show network segments. |
| 12 | Service | Show service areas. |
| 13 | Alarms | Show network alarms. |
| 14 | Metrics | Show performance metrics. |
| 15 | Impact | Show customer impact. |

### Ops Agent — `Telecom_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Outages | Are there critical network outages? |
| 2 | Network | Which segments have poor performance? |
| 3 | SLA | Show SLA breaches. |
| 4 | Tickets | Which trouble tickets are critical and open? |
| 5 | Wellness | Which technicians are fatigued? |
| 6 | Dispatch | Show field dispatch delays. |
| 7 | Escalations | Show ticket escalation issues. |
| 8 | RCA | Show RCA quality issues. |
| 9 | Alarms | Show critical network alarms. |
| 10 | Metrics | Show performance degradation. |
| 11 | Impact | Show critical customer impact. |
| 12 | Customer | Show customer satisfaction drops. |
| 13 | NMS | Show unacknowledged NMS alarms. |
| 14 | Trends | Show network alarm trend. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")